In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/final_year_project/Data/fraud_oracle_raw.csv')

Mounted at /content/drive


In [3]:
# Data cleaning
df = df.drop(columns='PolicyNumber')

malformed_rows = (
    (df['DayOfWeekClaimed'].astype(str) == '0') |
    (df['MonthClaimed'].astype(str) == '0')
)
df = df.loc[~malformed_rows].reset_index(drop=True)

df['AgeWasZero'] = (df['Age'] == 0).astype(int)
df.loc[df['Age'] == 0, 'Age'] = 16

print(f"Cleaned: {df.shape[0]} rows × {df.shape[1]} columns")

Cleaned: 15419 rows × 33 columns


In [4]:
# Midpoint conversions
midpoint_mappings = {
    'VehiclePrice': {
        'less than 20000': 10000, '20000 to 29000': 24500,
        '30000 to 39000': 34500, '40000 to 59000': 49500,
        '60000 to 69000': 64500, 'more than 69000': 75000
    },
    'Days_Policy_Accident': {
        'none': 0, '1 to 7': 4, '8 to 15': 11.5,
        '15 to 30': 22.5, 'more than 30': 35
    },
    'Days_Policy_Claim': {
        'none': 0, '8 to 15': 11.5,
        '15 to 30': 22.5, 'more than 30': 35
    },
    'PastNumberOfClaims': {
        'none': 0, '1': 1, '2 to 4': 3, 'more than 4': 5
    },
    'AgeOfVehicle': {
        'new': 0, '2 years': 2, '3 years': 3, '4 years': 4,
        '5 years': 5, '6 years': 6, '7 years': 7, 'more than 7': 9
    },
    'AgeOfPolicyHolder': {
        '16 to 17': 16.5, '18 to 20': 19, '21 to 25': 23,
        '26 to 30': 28, '31 to 35': 33, '36 to 40': 38,
        '41 to 50': 45.5, '51 to 65': 58, 'over 65': 70
    },
    'NumberOfSuppliments': {
        'none': 0, '1 to 2': 1.5, '3 to 5': 4, 'more than 5': 7
    },
    'AddressChange_Claim': {
        'no change': 0, 'under 6 months': 0.25,
        '1 year': 1, '2 to 3 years': 2.5, '4 to 8 years': 6
    },
    'NumberOfCars': {
        '1 vehicle': 1, '2 vehicles': 2, '3 to 4': 3.5,
        '5 to 8': 6.5, 'more than 8': 10
    },
}

for col, mapping in midpoint_mappings.items():
    df[f'{col}_num'] = df[col].map(mapping)

print(f"Midpoint features added: {len(midpoint_mappings)}")

Midpoint features added: 9


In [5]:
# Derived features
df['ClaimDelay'] = (df['Days_Policy_Claim_num'] - df['Days_Policy_Accident_num']).abs()
df['PriceToDeductibleRatio'] = df['VehiclePrice_num'] / df['Deductible'].replace(0, np.nan)
df['IsWeekendAccident'] = df['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)
df['LowPriceHighInsured'] = (
    df['VehiclePrice'].isin(['less than 20000', '20000 to 29000']) &
    (df['BasePolicy'] == 'All Perils')
).astype(int)
df['NoEvidence'] = (
    (df['PoliceReportFiled'] == 'No') &
    (df['WitnessPresent'] == 'No')
).astype(int)

df['Fault_PolicyType'] = df['Fault'] + '_' + df['PolicyType']
df['Area_PolicyType']  = df['AccidentArea'] + '_' + df['PolicyType']

print(f"Total columns: {df.shape[1]}")

Total columns: 49


In [6]:
# Save clean dataset
df.to_csv('/content/drive/MyDrive/final_year_project/Data/fraud_oracle_clean.csv', index=False)
print("Saved: fraud_oracle_clean.csv")

Saved: fraud_oracle_clean.csv
